### `etd_rk3.ipynb`
*Created: June 15, 2026*

This notebook implements the `ETD-RK3` method, a third-order exponential time differencing (ETD) method for solving systems of differential equations of the form $\frac{du}{dt} = Au + N(u,t)$, where $A$ is an $N \times N$ constant matrix and $N$ is a nonlinear function. 

In [3]:
using LinearAlgebra, LaTeXStrings, Printf, NBInclude, Statistics, UnPack
@nbinclude("phi_functions.ipynb")

In [2]:
function etd_rk3(A, f, u0; tspan::Tuple, Δt::Float64, p = nothing, ϵ::Float64 = 1e-3)
    """
    Solves the ODE system du/dt = Au + f(u,p,t), u(0) = u0 for t ∈ [tspan[1], tspan[2]]. 
    
    PARAMETERS
    ----------
    A :: N x N constant matrix (a scalar if N = 1)
    f :: Nonlinear function from R^N to R^N 
    u0 :: The initial condition (a vector of length N, or a scalar) 
    tspan :: The time interval over which to integrate the ODE. 
    Δt :: The time step (fixed)
    p :: Parameter for the nonlinear term `f` (if required) 
    
    RETURNS
    -------
    sol : Tuple 
        sol.u  :: the 
        sol.t  :: 
        sol.Δt ::
    
    """

    ################  INPUT VALIDATION ######################

    #If A is a scalar, u0 must be a scalar.  If A is a matrix, u0 must be a vector. 
    (A isa AbstractMatrix && u0 isa AbstractVector) || (A isa Number && u0 isa Number) || 
    throw(ArgumentError("Types of A and u0 are incompatible: typeof(A) = $(typeof(A)), typeof(u0) = $(typeof(u0))"))

    #If A is a matrix, it must be square and u0 must have compatible size 
    if A isa AbstractMatrix
        size(A, 1) == size(A, 2) || throw(ArgumentError("A must be a square matrix. Passed A with size $(size(A))"))
        size(A, 1) == length(u0) || throw(ArgumentError("Must have size(A,1) == size(A,2) = length(u0)." ))        
    end 

    tspan[1] < tspan[2] || throw(ArgumentError("t0 must be smaller than tf. Passed t0 = $(tspan[1]), tf = $(tspan[2])."))

    ################  TYPE CONVERSION ##########################
   
    #Get the eltype (if array) or type (if scalar) of A and u0
    A_type = A isa AbstractMatrix ? eltype(A) : typeof(A)
    u0_type = u0 isa AbstractVector ? eltype(u0) : typeof(u0) 

    T = promote_type(A_type, u0_type, Float64)       
    A = A isa AbstractMatrix ? Matrix{T}(A) : T(A)
    u0 = u0 isa AbstractVector ? Vector{T}(u0) : T(u0)
    
    ############################################################

    t0, tf = tspan 
    nsteps = Int(floor((tf - t0) / Δt))                      #Number of time steps 
    t = collect(range(t0, step = Δt, length = nsteps + 1))   #Times at which to record solution 

    u = [zero(u0) for _ in 1:nsteps+1]    #Vector to store solution iterates 
    u[1] = u0 

    #Compute some matrices

    Z = Δt*A
    Z_half = (Δt/2)*A 
    
    Φ₀, Φ₁, Φ₂, Φ₃ = phis(Z, 3; ϵ = ϵ)          #ϕ_0(Δt*A), ϕ_1(Δt*A), ϕ_2(Δt*A), ϕ_3(Δt*A)
    Φ₀_half, Φ₁_half = phis(Z_half, 1; ϵ = ϵ)   #ϕ_0(Δt*A/2), ϕ_1(Δt*A/2) 
    
    B₁ = Φ₁ - 3*Φ₂ + 4*Φ₃
    B₂ = 4*Φ₂ - 8*Φ₃
    B₃ = -Φ₂ + 4*Φ₃
    
    for n = 1:nsteps   #n+1 = 2,...nsteps + 1 
        
        N₁ = f(u[n], p, t[n])
        aₙ = Φ₀_half * u[n]  +  (Δt/2) * Φ₁_half * N₁
        
        N₂ = f(aₙ, p, t[n] + Δt/2)
        bₙ = Φ₀ * u[n]       +  Δt * Φ₁ * (2*N₂ - N₁)
        
        N₃ = f(bₙ, p, t[n] + Δt)

        u[n+1] = Φ₀*u[n] + Δt * (B₁*N₁ + B₂*N₂ + B₃*N₃) 

    end 
   
    return (u = u, t = t, Δt = Δt, p = p) 
end 

etd_rk3 (generic function with 1 method)